# Scraping nagłówków - jeden driver dla trzech źródeł

Cała logika siedzi w `src/scraping/`, notebook ją wywołuje i pokazuje wyniki.

| label | tytuł | moduł | technika |
|---|---|---|---|
| 0 | Niezależna | `scrape_newspapers.py` | async, HTML + CSS |
| 1 | OKO.press | `scrape_okopress.py` | sync, JSON `__NEXT_DATA__` |
| 2 | RP | `scrape_newspapers.py` | async, HTML + CSS |

Korpus zebrano jednorazowo i od tego czasu notebooka nie uruchamiano, dlatego komórki nie mają zapisanych wyników. Dokumentuje on metodę pobrania, nie odtwarza danych: serwisy zmieniają układ stron, więc selektory mogą już nie pasować. Punktem wejścia potoku jest `data/korpus_surowy.csv`.

Komórki poniżej pobierają domyślnie kilka stron. Parametry pełnego pobrania podano przy każdej z nich.


In [ ]:
# --- bootstrap: dodaj katalog projektu do sys.path (dziala z notebooks/ i z root) ---
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / 'config.py').exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))
from IPython.display import display

import config
import pandas as pd
print('DATA =', config.DATA)


## 1. OKO.press (klasa 1)

Scraper synchroniczny, czytający JSON z `__NEXT_DATA__`. Dopisuje przyrostowo do `config.OKO_CSV` i pomija adresy już zapisane, więc przerwany przebieg wznawia się bez duplikatów. Parametr `since` zatrzymuje go na dacie granicznej.

Pełne pobranie: `scrape_okopress(since='2022-01-01')` bez `max_pages`.


In [ ]:
from src.scraping import scrape_okopress

scrape_okopress(max_pages=3)


## 2. Gazety: Niezależna (0) i RP (2)

Scraper asynchroniczny, czytający HTML selektorami CSS. `scrape_sources` przyjmuje listę źródeł i zapisuje wspólny `config.NEWSPAPERS_CSV`. Pętla zdarzeń w Jupyterze już działa, więc wywołanie idzie przez `await`, nie przez `asyncio.run`.

Pełne pobranie: `pages=1000`.


In [ ]:
from src.scraping import scrape_sources

df_papers = await scrape_sources(['niezalezna', 'rp'], pages=2)
df_papers.head() if df_papers is not None else 'brak danych'


## 3. Podgląd wyników


In [ ]:
oko = pd.read_csv(config.OKO_CSV)
print('OKO.press:', len(oko), 'wierszy')
display(oko[['text', 'kategoria', 'date']].head())


In [ ]:
try:
    papers = pd.read_csv(config.NEWSPAPERS_CSV)
    print('Gazety:', len(papers), 'wierszy')
    print(papers['label'].value_counts())
    display(papers.head())
except FileNotFoundError:
    print('Brak', config.NEWSPAPERS_CSV, '- uruchom sekcję 2.')
